# Imports

In [24]:
import pandas as pd
import numpy as np

# Get and Format Data

In [56]:
train_set = pd.read_csv("datasets/train_set.csv")
test_set = pd.read_csv("datasets/test_set.csv")

In [57]:
train_set = train_set.rename(columns=lambda x: x[2:] if x[1]=='_' else x)
test_set = test_set.rename(columns=lambda x: x[2:] if x[1]=='_' else x)

to_drop = [col for col in test_set.columns if 'lag0' in col]
train_set = train_set.drop(to_drop, axis=1)
test_set = test_set.drop(to_drop, axis=1)
train_set['class'] = train_set['class'].apply(lambda x: 1 if x != 0 else 0)
test_set['class'] = test_set['class'].apply(lambda x: 1 if x != 0 else 0)


In [58]:
indices_to_drop = train_set[train_set['class'] == 1].sample(n=5000).index
train_set = train_set.drop(indices_to_drop)

indices_to_drop = test_set[test_set['class'] == 1].sample(n=5000).index
test_set = test_set.drop(indices_to_drop)


In [59]:

print(train_set['class'].describe())
print(test_set['class'].describe())

count    9931.000000
mean        0.502165
std         0.500020
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: class, dtype: float64
count    9944.000000
mean        0.501911
std         0.500021
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: class, dtype: float64


# Feature Selection

In [60]:
all_features = train_set.columns.drop(['system:index', 'class', 'latitude', 'longitude', '.geo'])
print(all_features)

Index(['NBR_delta_lag4', 'NBR_lag4', 'NDMI_delta_lag4', 'NDMI_lag4',
       'NDVI_delta_lag4', 'NDVI_lag4', 'SR_B4_delta_lag4', 'SR_B4_lag4',
       'SR_B5_delta_lag4', 'SR_B5_lag4', 'SR_B6_delta_lag4', 'SR_B6_lag4',
       'SR_B7_delta_lag4', 'SR_B7_lag4', 'NBR_delta_lag3', 'NBR_lag3',
       'NDMI_delta_lag3', 'NDMI_lag3', 'NDVI_delta_lag3', 'NDVI_lag3',
       'SR_B4_delta_lag3', 'SR_B4_lag3', 'SR_B5_delta_lag3', 'SR_B5_lag3',
       'SR_B6_delta_lag3', 'SR_B6_lag3', 'SR_B7_delta_lag3', 'SR_B7_lag3',
       'NBR_delta_lag2', 'NBR_lag2', 'NDMI_delta_lag2', 'NDMI_lag2',
       'NDVI_delta_lag2', 'NDVI_lag2', 'SR_B4_delta_lag2', 'SR_B4_lag2',
       'SR_B5_delta_lag2', 'SR_B5_lag2', 'SR_B6_delta_lag2', 'SR_B6_lag2',
       'SR_B7_delta_lag2', 'SR_B7_lag2', 'NBR_delta_lag1', 'NBR_lag1',
       'NDMI_delta_lag1', 'NDMI_lag1', 'NDVI_delta_lag1', 'NDVI_lag1',
       'SR_B4_delta_lag1', 'SR_B4_lag1', 'SR_B5_delta_lag1', 'SR_B5_lag1',
       'SR_B6_delta_lag1', 'SR_B6_lag1', 'SR_B7_delta_lag

In [62]:
X_train = train_set[all_features]
X_test = test_set[all_features]
y_train = train_set['class']
y_test = test_set['class']

# Logistic Regression

In [63]:
print(y_train.unique())

[1 0]


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import cross_validate

pipe = Pipeline([('transformer', PowerTransformer('yeo-johnson')), ('model', LogisticRegressionCV(cv=5, Cs=1, l1_ratios=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], l1_ratio=0.5, solver='saga', random_state=1))])
results = cross_validate(
    pipe, 
    X_train, 
    y_train, 
    cv=5, 
    scoring='f1', 
    return_train_score=True  # Crucial for detecting overfitting
)

/opt/anaconda3/envs/deforestationenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/opt/anaconda3/envs/deforestationenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: 1.

# Predictions

In [16]:
from sklearn.metrics import f1_score

y_pred_train = pipe.predict(X_train)
y_pred_val = pipe.predict(X_val)

train_error = f1_score(y_train, y_pred_train)
test_error = f1_score(y_val, y_pred_val)
print(f'Train f1: {train_error}')
print(f'Test f1: {test_error}')
print(f'Overfitting Ratio: {(train_error-test_error)}')

NameError: name 'pipe' is not defined